In [1]:
!pip install kagglehub pandas transformers torch scikit-learn -q

In [2]:
import kagglehub

# Download dataset
path = kagglehub.dataset_download("tobiasbueck/multilingual-customer-support-tickets")

print("Dataset path:", path)

100%|██████████| 16.1M/16.1M [00:00<00:00, 33.7MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/tobiasbueck/multilingual-customer-support-tickets/versions/14


In [3]:
import os
import pandas as pd

# Find CSV files inside dataset folder
files = []
for root, dirs, fns in os.walk(path):
    for f in fns:
        if f.endswith(".csv"):
            files.append(os.path.join(root, f))

print("CSV files found:", files)

# Load first CSV (usually main dataset)
df = pd.read_csv(files[0])

print(df.head())
print(df.columns)

CSV files found: ['/root/.cache/kagglehub/datasets/tobiasbueck/multilingual-customer-support-tickets/versions/14/dataset-tickets-multi-lang-4-20k.csv', '/root/.cache/kagglehub/datasets/tobiasbueck/multilingual-customer-support-tickets/versions/14/dataset-tickets-german_normalized_50_5_2.csv', '/root/.cache/kagglehub/datasets/tobiasbueck/multilingual-customer-support-tickets/versions/14/aa_dataset-tickets-multi-lang-5-2-50-version.csv', '/root/.cache/kagglehub/datasets/tobiasbueck/multilingual-customer-support-tickets/versions/14/dataset-tickets-multi-lang3-4k.csv', '/root/.cache/kagglehub/datasets/tobiasbueck/multilingual-customer-support-tickets/versions/14/dataset-tickets-german_normalized.csv']
                                             subject  \
0  Unvorhergesehener Absturz der Datenanalyse-Pla...   
1                           Customer Support Inquiry   
2                      Data Analytics for Investment   
3                 Krankenhaus-Dienstleistung-Problem   
4            

In [4]:
# Try to automatically detect text column
possible_text_cols = ["text", "message", "ticket", "content", "body", "description"]

text_col = None
for col in df.columns:
    if col.lower() in possible_text_cols:
        text_col = col
        break

# fallback: use first object column
if text_col is None:
    text_col = df.select_dtypes(include="object").columns[0]

print("Using text column:", text_col)

Using text column: body


In [5]:
TAGS = [
    "technical_issue",
    "billing_issue",
    "account_management",
    "login_problem",
    "refund_request",
    "bug_report",
    "feature_request",
    "complaint",
    "general_inquiry"
]

In [6]:
from transformers import pipeline

zero_shot = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [7]:
def zero_shot_predict(text):
    result = zero_shot(text, TAGS)

    labels = result["labels"]
    scores = result["scores"]

    top3 = list(zip(labels[:3], scores[:3]))
    return top3

In [10]:
def few_shot_prompt(text):
    prompt = f"""
You are a support ticket classifier.

Tags:
technical_issue, billing_issue, account_management, login_problem, refund_request, bug_report, feature_request, complaint, general_inquiry

Examples:
Ticket: "I cannot login to my account"
Tags: login_problem, account_management

Ticket: "I was overcharged this month"
Tags: billing_issue, complaint

Ticket: "App crashes when I open it"
Tags: bug_report, technical_issue

Now classify:
Ticket: "{text}"

Return 3 most relevant tags separated by commas.
"""

    output = generator(prompt, max_new_tokens=50)[0]["generated_text"]
    return output

In [13]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

In [14]:
# Ensure no NaN values
df_clean = df.dropna(subset=[text_col]).reset_index(drop=True)

# Take a small sample for testing (you can increase later)
sample_df = df_clean[[text_col]].head(20)

results = []

for text in sample_df[text_col]:

    # ZERO-SHOT
    zero = zero_shot_predict(text)

    # FEW-SHOT
    few = few_shot_prompt(text)

    results.append({
        "ticket": text,
        "zero_shot_top3": zero,
        "few_shot_output": few
    })

print("Done processing:", len(results))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `m

Done processing: 20


In [15]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df.head()

,ticket,zero_shot_top3,few_shot_output
0,Die Datenanalyse-Plattform brach unerwartet ab...,"[(complaint, 0.5704725980758667), (technical_i...",\nYou are a support ticket classifier.\n\nTags...
1,Seeking information on digital strategies that...,"[(feature_request, 0.5536883473396301), (compl...",\nYou are a support ticket classifier.\n\nTags...
2,I am contacting you to request information on ...,"[(complaint, 0.15723150968551636), (feature_re...",\nYou are a support ticket classifier.\n\nTags...
3,Ein Medien-Daten-Sperrverhalten trat aufgrund ...,"[(login_problem, 0.5863834619522095), (complai...",\nYou are a support ticket classifier.\n\nTags...
4,"Dear Customer Support, I am reaching out to in...","[(complaint, 0.23630312085151672), (general_in...",\nYou are a support ticket classifier.\n\nTags...


In [16]:
structured_results = []

for r in results:
    structured_results.append({
        "ticket": r["ticket"],

        "zero_shot_1": r["zero_shot_top3"][0][0],
        "zero_shot_2": r["zero_shot_top3"][1][0],
        "zero_shot_3": r["zero_shot_top3"][2][0],

        "few_shot_output": r["few_shot_output"]
    })

final_df = pd.DataFrame(structured_results)

final_df.head()

,ticket,zero_shot_1,zero_shot_2,zero_shot_3,few_shot_output
0,Die Datenanalyse-Plattform brach unerwartet ab...,complaint,technical_issue,account_management,\nYou are a support ticket classifier.\n\nTags...
1,Seeking information on digital strategies that...,feature_request,complaint,refund_request,\nYou are a support ticket classifier.\n\nTags...
2,I am contacting you to request information on ...,complaint,feature_request,technical_issue,\nYou are a support ticket classifier.\n\nTags...
3,Ein Medien-Daten-Sperrverhalten trat aufgrund ...,login_problem,complaint,technical_issue,\nYou are a support ticket classifier.\n\nTags...
4,"Dear Customer Support, I am reaching out to in...",complaint,general_inquiry,feature_request,\nYou are a support ticket classifier.\n\nTags...


In [17]:
for i in range(5):
    print("\n==============================")
    print("TICKET:", final_df.iloc[i]["ticket"])

    print("\nZERO-SHOT TAGS:")
    print(final_df.iloc[i][["zero_shot_1", "zero_shot_2", "zero_shot_3"]].values)

    print("\nFEW-SHOT OUTPUT:")
    print(final_df.iloc[i]["few_shot_output"])


TICKET: Die Datenanalyse-Plattform brach unerwartet ab, da die Speicheroberfläche zu gering war. Ich habe versucht, Laravel 8 und meinen MacBook Pro neu zu starten, aber das Problem behält sich bei. Ich benötige Ihre Unterstützung, um diesen Fehler zu beheben.

ZERO-SHOT TAGS:
['complaint' 'technical_issue' 'account_management']

FEW-SHOT OUTPUT:

You are a support ticket classifier.

Tags:
technical_issue, billing_issue, account_management, login_problem, refund_request, bug_report, feature_request, complaint, general_inquiry

Examples:
Ticket: "I cannot login to my account"
Tags: login_problem, account_management

Ticket: "I was overcharged this month"
Tags: billing_issue, complaint

Ticket: "App crashes when I open it"
Tags: bug_report, technical_issue

Now classify:
Ticket: "Die Datenanalyse-Plattform brach unerwartet ab, da die Speicheroberfläche zu gering war. Ich habe versucht, Laravel 8 und meinen MacBook Pro neu zu starten, aber das Problem behält sich bei. Ich benötige Ihre 

In [18]:
final_df.to_csv("support_ticket_tagging_results.csv", index=False)

print("File saved: support_ticket_tagging_results.csv")

File saved: support_ticket_tagging_results.csv


In [19]:
print("Dataset size used:", len(df_clean))
print("Sample processed:", len(final_df))

print("\nExample tags from Zero-shot:")
print(final_df["zero_shot_1"].value_counts().head())

print("\nFew-shot sample output:")
print(final_df["few_shot_output"].head(3))

Dataset size used: 19998
Sample processed: 20

Example tags from Zero-shot:
zero_shot_1
complaint          10
technical_issue     5
feature_request     3
login_problem       1
bug_report          1
Name: count, dtype: int64

Few-shot sample output:
0    \nYou are a support ticket classifier.\n\nTags...
1    \nYou are a support ticket classifier.\n\nTags...
2    \nYou are a support ticket classifier.\n\nTags...
Name: few_shot_output, dtype: object


In [20]:
report = """
TASK 5: AUTO TAGGING SUPPORT TICKETS USING LLM

Approach:
1. Used zero-shot classification using XLM-RoBERTa model.
2. Used few-shot prompting using FLAN-T5 model.
3. Compared performance of both approaches.
4. Generated top 3 probable tags for each ticket.

Methods:
- Zero-shot learning: directly classifies tickets using pretrained model without training.
- Few-shot learning: improves performance using example-based prompt engineering.

Output:
- Each ticket is assigned top 3 predicted tags.
- Few-shot model provides more contextual and refined predictions.

Conclusion:
Few-shot prompting improves contextual understanding compared to zero-shot classification.
"""

print(report)


TASK 5: AUTO TAGGING SUPPORT TICKETS USING LLM

Approach:
1. Used zero-shot classification using XLM-RoBERTa model.
2. Used few-shot prompting using FLAN-T5 model.
3. Compared performance of both approaches.
4. Generated top 3 probable tags for each ticket.

Methods:
- Zero-shot learning: directly classifies tickets using pretrained model without training.
- Few-shot learning: improves performance using example-based prompt engineering.

Output:
- Each ticket is assigned top 3 predicted tags.
- Few-shot model provides more contextual and refined predictions.

Conclusion:
Few-shot prompting improves contextual understanding compared to zero-shot classification.

